# EX-08 Flat to Dome · straight pictures in the sweet spot

**A film or a live piece · for slides, photos, title cards and 16:9 video**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dcuartielles/wisdome-360/blob/main/docs/examples/ex08-flat-to-dome.ipynb)

![A 16:9 test slide put through the fisheye: it sits just below the centre of the master, bent so the dome straightens it](https://dcuartielles.github.io/wisdome-360/examples/images/ex08-flat-to-dome.jpg)

| File | What it is |
|---|---|
| [`flat2dome.html`](https://dcuartielles.github.io/wisdome-360/examples/files/EX-08-flat-to-dome/flat2dome.html) | Open it in Chrome, drop in a picture or a video, move the screen with sliders, save a PNG master or record a video. No install |
| [`flat2dome.py`](https://dcuartielles.github.io/wisdome-360/examples/files/EX-08-flat-to-dome/flat2dome.py) | Stills and image sequences from the command line. Only numpy and Pillow |
| This notebook | The method, step by step, on a test slide or your own picture |

## The problem

A dome master is a fisheye photograph of the dome. Paste a flat 16:9 picture into it and two things go wrong:

1. **It lands in the wrong place.** The centre of a Wisdome master is the dome's zenith, 64° up in front of the audience (the dome is tilted 26°, raised at the back). A picture pasted in the middle hangs above everyone's heads.
2. **It bends.** The dome undoes the fisheye. A picture that was never put *through* a fisheye comes out of the dome curved and stretched.

## The fix: apply the fisheye lens

Treat the picture as a **flat screen floating in the room**, in front of the audience, and photograph that screen with the same 180° fisheye the dome uses. The dome bends the photograph back, and from the seats the screen looks flat and straight.

```text
   audience ──▶  virtual screen, 60° wide, centred 40° up (the sweet spot)
       │
       └─ 180° fisheye camera, leaning 26° to the front like the dome
                     │
                     ▼
              dome master (square fisheye file)
```

In [ ]:
%pip install -q numpy pillow

In [ ]:
import math
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

## The method, in three steps

For every pixel of the dome master:

1. **pixel → direction on the dome:** equidistant fisheye, so the angle from the dome's zenith is the distance from the centre × 90°;
2. **dome → room:** tilt that direction 26°;
3. **room → screen:** intersect the direction with the virtual screen, like a pinhole camera, and read the colour of the picture there.

In [ ]:
def screen_uv(n, width, elev, az, tilt, aspect, curved, fov=180.0):
    """For every master pixel: where it lands on the flat picture (u, v in 0..1)."""
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float64)
    c = (n - 1) / 2.0
    dx, dy = (xx - c) / (n / 2.0), (yy - c) / (n / 2.0)
    r = np.hypot(dx, dy)
    # 1. master pixel -> direction on the DOME (front at the bottom)
    th = r * math.radians(fov / 2.0)
    a = np.arctan2(dx, dy)
    lx, ly, lz = np.sin(th) * np.sin(a), np.sin(th) * np.cos(a), np.cos(th)
    # 2. dome -> ROOM (x right, y front, z up): the dome's zenith leans `tilt` to the front
    ct, st = math.cos(math.radians(tilt)), math.sin(math.radians(tilt))
    d = np.stack([lx, ly * ct + lz * st, -ly * st + lz * ct], -1)
    # 3. the screen: centre direction f, right vector rr, up vector uu
    e, h = math.radians(elev), math.radians(az)
    f = np.array([math.cos(e) * math.sin(h), math.cos(e) * math.cos(h), math.sin(e)])
    rr = np.cross(f, [0.0, 0.0, 1.0]); rr /= np.linalg.norm(rr)
    uu = np.cross(rr, f)
    half_w = math.radians(width) / 2.0
    df, dr, du = d @ f, d @ rr, d @ uu
    if curved:   # a cylinder round the audience, for very wide pictures
        x = np.arctan2(dr, df) / half_w
        y = du / np.maximum(np.hypot(dr, df), 1e-9) / (half_w / aspect)
        ok = (r <= 1.0) & (np.hypot(dr, df) > 1e-6)
    else:        # a flat screen: intersect each ray with the plane
        k = np.where(df > 1e-6, df, np.nan)
        x = dr / k / math.tan(half_w)
        y = du / k / (math.tan(half_w) / aspect)
        ok = (r <= 1.0) & (df > 1e-6)
    u, v = (x + 1) / 2, (1 - y) / 2                  # image rows go DOWN
    inside = ok & (u >= 0) & (u <= 1) & (v >= 0) & (v <= 1)
    return u, v, inside


def flat2dome(img, n=2048, width=60, elev=40, az=0, tilt=26, curved=False):
    """A flat picture -> a dome master with it on a virtual screen in the room."""
    src = np.asarray(img.convert("RGB")).astype(np.float32)
    H, W = src.shape[:2]
    u, v, inside = screen_uv(n, width, elev, az, tilt, W / H, curved)
    sx = np.clip(np.nan_to_num(u) * (W - 1), 0, W - 1)
    sy = np.clip(np.nan_to_num(v) * (H - 1), 0, H - 1)
    x0, y0 = np.floor(sx).astype(int), np.floor(sy).astype(int)
    x1, y1 = np.minimum(x0 + 1, W - 1), np.minimum(y0 + 1, H - 1)
    fx, fy = (sx - x0)[..., None], (sy - y0)[..., None]
    out = (src[y0, x0] * (1 - fx) * (1 - fy) + src[y0, x1] * fx * (1 - fy)
           + src[y1, x0] * (1 - fx) * fy + src[y1, x1] * fx * fy)
    out[~inside] = 0
    return Image.fromarray(out.astype(np.uint8))

## A test slide

Dark, with big type and a grid of straight lines, so you can see whether they stay straight. To use your own picture instead, see the next cell.

In [ ]:
def test_slide(w=1920, h=1080):
    im = Image.new("RGB", (w, h), (12, 16, 28))
    d = ImageDraw.Draw(im)
    for x in range(0, w + 1, w // 8):
        d.line([(x, 0), (x, h)], fill=(60, 80, 120), width=6)
    for y in range(0, h + 1, h // 6):
        d.line([(0, y), (w, y)], fill=(60, 80, 120), width=6)
    d.rectangle([3, 3, w - 4, h - 4], outline=(255, 126, 77), width=10)
    font = ImageFont.load_default(size=h // 6)
    d.text((w / 2, h * 0.42), "Hello, dome", font=font, fill=(242, 245, 250), anchor="mm")
    small = ImageFont.load_default(size=h // 14)
    d.text((w / 2, h * 0.70), "a flat 16:9 slide", font=small, fill=(255, 216, 77), anchor="mm")
    return im

def rim(master):
    """For looking only: draw the edge of the circle, which is black in a real master."""
    im = master.copy()
    ImageDraw.Draw(im).ellipse([0, 0, im.width - 1, im.height - 1], outline=(70, 90, 125), width=2)
    return im

slide = test_slide()
display(slide.resize((640, 360)))

**Your own picture.** In Colab, run the next cell and choose a file. In Jupyter, put the file next to the notebook and write `slide = Image.open("my_slide.png")`.

In [ ]:
USE_MY_OWN = False

if USE_MY_OWN:
    from google.colab import files
    name = next(iter(files.upload()))
    slide = Image.open(name)

## The right way, and the wrong way

Left: the slide on a 60°-wide screen at the sweet spot, put through the fisheye. Right: the same slide pasted into the middle of the master. On the dome, the left one is flat and straight in front of you; the right one hangs overhead and bends.

In [ ]:
right = flat2dome(slide, n=1024)
wrong = Image.new("RGB", (1024, 1024))
s = slide.resize((520, int(520 * slide.height / slide.width)))
wrong.paste(s, ((1024 - s.width) // 2, (1024 - s.height) // 2))
both = Image.new("RGB", (2048 + 32, 1024))
both.paste(rim(right), (0, 0)); both.paste(rim(wrong), (1024 + 32, 0))
display(both.resize((1040, 512)))

## How big, and where

| Choice | Recommended | Why |
|---|---|---|
| Centre | **40° up, straight ahead** | the sweet spot |
| Width | **50–70°** (60° is a good start) | a 16:9 screen 60° wide is 36° tall and spans 22°–58° up: inside the comfortable zone |
| Shape | flat; **curved** only for very wide pictures | flat keeps straight lines straight; curved avoids stretched edges when wider than about 90° |
| Text | cap height ≥ **1.5°** | on a 60° screen that is ≥ 4 % of the slide height, about 45 px on a 1080 px slide. See [EX-09](https://dcuartielles.github.io/wisdome-360/examples/ex09-type-size/) |

**Resolution budget.** A 60° screen gets about **680 px across in a 2048 master**. A 1920 px slide is shrunk about three times, so use big type, thick lines and no fine detail.

Change the numbers and look:

In [ ]:
display(rim(flat2dome(slide, n=768, width=90, elev=35)))                 # wide, flat: stretched edges
display(rim(flat2dome(slide, n=768, width=90, elev=35, curved=True)))    # wide, curved

## Save a master for the dome

2048² is the [delivery size](https://dcuartielles.github.io/wisdome-360/reference/delivery-spec/). Open the PNG in the [Dome Simulator](https://dcuartielles.github.io/wisdome-360/tools/dome-simulator.html) and look at it from the back row (key `4`).

In [ ]:
out = flat2dome(slide, n=2048)
out.save("slide_dome_2048.png")
print("saved slide_dome_2048.png")

## A whole video: one ffmpeg line

ffmpeg's `v360` filter does the same projection. For a 16:9 film on a 60°-wide screen at the sweet spot:

```bash
ffmpeg -i film.mp4 -filter_complex \
 "color=black:s=2048x2048:r=30[bg];[0:v]v360=input=flat:output=fisheye:ih_fov=60:iv_fov=36.0:h_fov=180:v_fov=180:pitch=24:w=2048:h=2048:interp=lanczos:alpha_mask=1,format=rgba[v];[bg][v]overlay=shortest=1,format=yuv420p" \
 -c:v libx264 -profile:v high -b:v 60M -maxrate 60M -bufsize 120M -r 30 -c:a copy film_dome_2048.mp4
```

The two numbers to change:

- `pitch` = **90 − 26 − elevation**: 24 puts the centre at 40° up.
- `iv_fov` = **2 · atan(tan(ih_fov / 2) · height / width)**. For a 60°-wide 16:9 picture that is 36.0, *not* 33.75: a flat screen's angles don't scale linearly.

This command agrees with `flat2dome()` to within a pixel. Keep `yaw` at 0: it turns around the dome's own axis, not the room's. For a screen off to one side, use the HTML tool or `--az` in `flat2dome.py`.

## Inside a 3-D tool

Put the picture on a plane in front of the dome camera, sized by angle: width `2 · D · tan(W/2)` at distance `D`, height = width ÷ aspect, centred at 40° up and facing the audience, with an **emission** (unlit) material so it keeps its own colours. For D = 10 and W = 60°: 11.55 × 6.50. [EX-01](https://dcuartielles.github.io/wisdome-360/examples/ex01-drift/) has the Blender camera and `place_at()`, [EX-05](https://dcuartielles.github.io/wisdome-360/examples/ex05-touchdesigner/) the TouchDesigner one.

## What usually breaks

| Symptom | Cause |
|---|---|
| The picture is overhead and curved | It was pasted into the master instead of put through the fisheye |
| The whole dome greys out | A white or bright slide lights the dome by reflection. Keep slides dark |
| The edges are badly stretched | The screen is wider than about 90°. Make it narrower, or use `curved=True` |
| Text is mush | Smaller than 1.5° cap height, or thin weights. See EX-09 |
| It looks bent from the aisle seats | Normal: the screen is exactly flat only from the centre. Keep it near the sweet spot and not too wide |